# Hu et al. TransRAC Pose Adaptation

Hu et al. propose TransRAC for repetitive action counting using fixed-length video clips, multi-scale temporal correlation, and density-map regression. This notebook adapts those ideas to the CoreSet pose dataset by using interpolated MediaPipe landmarks instead of RGB video frames.

The input is `data/interpolated_json/<category>/*.json`. The output is `data/hu_transrac_features/<category>/*.npz`, where each file contains a 64-frame normalized pose sequence, multi-scale temporal correlation maps, and a count/density target.


## Setup, Paths, and Hu-Style Parameters

TransRAC uses a fixed temporal length and multi-scale temporal windows. The official implementation commonly uses 64 frames and multi-scale windows such as 1, 4, and 8. This cell sets those parameters for pose features.


In [ ]:
import json
import math
from pathlib import Path
from collections import Counter

import numpy as np
from scipy.signal import find_peaks
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_BASE = PROJECT_ROOT / 'data' / 'interpolated_json'
OUTPUT_BASE = PROJECT_ROOT / 'data' / 'hu_transrac_features'
MANIFEST_PATH = OUTPUT_BASE / 'manifest.json'

CATEGORIES = ['bench_press', 'pull_up', 'push_up', 'squat']
CATEGORY_TO_INDEX = {name: idx for idx, name in enumerate(CATEGORIES)}
EXPECTED_LANDMARKS = 33
FEATURES_PER_LANDMARK = 4
NUM_FRAMES = 64
SCALES = (1, 4, 8)
EPS = 1e-8

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)


## Load Pose JSON into a Numeric Tensor

Each JSON is converted into a tensor with shape `(frames, 33, 4)`. The four landmark values are `[x, y, z, visibility]`. This keeps the interpolated dataset format intact while giving the Hu-style pipeline a consistent numeric input.


In [ ]:
def landmark_to_vector(landmark):
    if landmark is None:
        return [0.0, 0.0, 0.0, 0.0]

    if isinstance(landmark, dict):
        values = [
            landmark.get('x', 0.0),
            landmark.get('y', 0.0),
            landmark.get('z', 0.0),
            landmark.get('visibility', landmark.get('v', 0.0)),
        ]
    else:
        values = list(landmark[:FEATURES_PER_LANDMARK])
        values += [0.0] * (FEATURES_PER_LANDMARK - len(values))

    clean = []
    for value in values:
        try:
            value = float(value)
        except (TypeError, ValueError):
            value = 0.0
        clean.append(value if math.isfinite(value) else 0.0)
    return clean


def load_pose_json(path):
    with path.open('r') as f:
        data = json.load(f)

    frames = data.get('frames', data) if isinstance(data, dict) else data
    frames = sorted(frames, key=lambda frame: frame.get('frame_index', 0))

    tensor = np.zeros((len(frames), EXPECTED_LANDMARKS, FEATURES_PER_LANDMARK), dtype=np.float32)
    for frame_idx, frame in enumerate(frames):
        landmarks = frame.get('landmarks') if isinstance(frame, dict) else None
        landmarks = landmarks or []
        for lm_idx in range(min(EXPECTED_LANDMARKS, len(landmarks))):
            tensor[frame_idx, lm_idx] = landmark_to_vector(landmarks[lm_idx])

    metadata = data if isinstance(data, dict) else {}
    return tensor, metadata


## Normalize and Resample Pose Sequences

The RGB TransRAC model extracts video features with a backbone. For skeleton data, the equivalent input feature is a normalized pose vector. Each frame is centered around the hip midpoint and scaled by a stable body-size estimate, then resampled to 64 frames so every sample has the same temporal length.


In [ ]:
def normalize_pose_tensor(tensor):
    coords = tensor[:, :, :3].astype(np.float32).copy()
    visibility = np.clip(tensor[:, :, 3:4].astype(np.float32), 0.0, 1.0)

    hip_center = (coords[:, 23:24, :] + coords[:, 24:25, :]) / 2.0
    shoulder_center = (coords[:, 11:12, :] + coords[:, 12:13, :]) / 2.0
    center = np.where(np.isfinite(hip_center).all(axis=2, keepdims=True), hip_center, shoulder_center)
    coords = coords - center

    shoulder_width = np.linalg.norm(coords[:, 11, :] - coords[:, 12, :], axis=1)
    hip_width = np.linalg.norm(coords[:, 23, :] - coords[:, 24, :], axis=1)
    body_scale = np.maximum(shoulder_width, hip_width)
    valid_scale = body_scale[np.isfinite(body_scale) & (body_scale > EPS)]
    fallback_scale = float(np.median(valid_scale)) if valid_scale.size else 1.0
    body_scale = np.where(body_scale > EPS, body_scale, fallback_scale).reshape(-1, 1, 1)

    coords = coords / np.maximum(body_scale, EPS)
    normalized = np.concatenate([coords, visibility], axis=2)
    return np.nan_to_num(normalized, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def resample_sequence(sequence, target_frames=NUM_FRAMES):
    if len(sequence) == target_frames:
        return sequence.astype(np.float32)
    if len(sequence) == 0:
        return np.zeros((target_frames, EXPECTED_LANDMARKS, FEATURES_PER_LANDMARK), dtype=np.float32)
    if len(sequence) == 1:
        return np.repeat(sequence, target_frames, axis=0).astype(np.float32)

    source_x = np.linspace(0.0, 1.0, len(sequence))
    target_x = np.linspace(0.0, 1.0, target_frames)
    flat = sequence.reshape(len(sequence), -1)
    resampled = np.empty((target_frames, flat.shape[1]), dtype=np.float32)
    for feature_idx in range(flat.shape[1]):
        resampled[:, feature_idx] = np.interp(target_x, source_x, flat[:, feature_idx])
    return resampled.reshape(target_frames, EXPECTED_LANDMARKS, FEATURES_PER_LANDMARK)


def flatten_pose_sequence(sequence):
    return sequence.reshape(sequence.shape[0], EXPECTED_LANDMARKS * FEATURES_PER_LANDMARK).astype(np.float32)


## Multi-Scale Temporal Correlation Maps

TransRAC represents repetition by comparing frames across time at multiple temporal scales. This pose adaptation builds cosine self-similarity maps from the normalized pose features after temporal smoothing with windows of 1, 4, and 8 frames. The output has shape `(3, 64, 64)`.


In [ ]:
def moving_average_features(features, window):
    if window <= 1:
        return features.astype(np.float32)

    pad_left = window // 2
    pad_right = window - 1 - pad_left
    padded = np.pad(features, ((pad_left, pad_right), (0, 0)), mode='edge')
    kernel = np.ones(window, dtype=np.float32) / float(window)
    smoothed = np.empty_like(features, dtype=np.float32)
    for feature_idx in range(features.shape[1]):
        smoothed[:, feature_idx] = np.convolve(padded[:, feature_idx], kernel, mode='valid')
    return smoothed


def cosine_correlation_matrix(features):
    centered = features - features.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(centered, axis=1, keepdims=True)
    normalized = centered / np.maximum(norms, EPS)
    matrix = normalized @ normalized.T
    return np.clip(matrix, -1.0, 1.0).astype(np.float32)


def build_multiscale_correlation(features, scales=SCALES):
    matrices = []
    for scale in scales:
        scaled_features = moving_average_features(features, scale)
        matrices.append(cosine_correlation_matrix(scaled_features))
    return np.stack(matrices, axis=0).astype(np.float32)


## Count and Density Target

Hu et al. use fine-grained cycle annotations to train a density-map regressor. This dataset does not currently include manual cycle boundaries, so this cell uses `rep_count` if it exists. If no count exists, it creates a pseudo-count from pose motion peaks and turns the estimated cycle centers into a 64-value density target.


In [ ]:
def exercise_motion_signal(sequence, category):
    y = sequence[:, :, 1]
    if category == 'bench_press':
        return (y[:, 15] + y[:, 16]) / 2.0
    if category in {'pull_up', 'push_up', 'squat'}:
        return (y[:, 11] + y[:, 12]) / 2.0
    return y[:, 12]


def smooth_signal(signal, window=5):
    if len(signal) < window:
        return signal.astype(np.float32)
    kernel = np.ones(window, dtype=np.float32) / float(window)
    return np.convolve(signal, kernel, mode='same').astype(np.float32)


def infer_pseudo_count_and_centers(sequence, category):
    signal = smooth_signal(exercise_motion_signal(sequence, category))
    std = float(np.std(signal))
    if std < EPS:
        return 0.0, []

    prominence = max(0.05 * std, 0.01)
    distance = max(4, NUM_FRAMES // 16)
    peaks, _ = find_peaks(signal, prominence=prominence, distance=distance)
    valleys, _ = find_peaks(-signal, prominence=prominence, distance=distance)
    centers = peaks if len(peaks) >= len(valleys) else valleys
    return float(len(centers)), centers.astype(int).tolist()


def density_from_centers(count, centers, length=NUM_FRAMES):
    density = np.zeros(length, dtype=np.float32)
    if count <= 0:
        return density

    if not centers:
        density[:] = count / float(length)
        return density

    x = np.arange(length, dtype=np.float32)
    sigma = max(length / (len(centers) * 8.0), 1.0)
    for center in centers:
        density += np.exp(-0.5 * ((x - float(center)) / sigma) ** 2).astype(np.float32)

    total = float(density.sum())
    if total > EPS:
        density *= count / total
    return density.astype(np.float32)


def build_density_target(metadata, sequence, category):
    stored_count = metadata.get('rep_count') or metadata.get('count') or metadata.get('repetition_count')
    if stored_count is not None:
        count = float(stored_count)
        centers = []
        source = 'metadata'
    else:
        count, centers = infer_pseudo_count_and_centers(sequence, category)
        source = 'pose_peak_pseudo_label'
    return density_from_centers(count, centers), count, centers, source


## Process the Interpolated JSON Dataset

This cell applies the Hu-style pose transformation to every interpolated JSON file. It saves compact `.npz` files so the original JSON dataset remains readable and unchanged.


In [ ]:
def process_one_file(json_path, category):
    raw_tensor, metadata = load_pose_json(json_path)
    normalized = normalize_pose_tensor(raw_tensor)
    sequence = resample_sequence(normalized, NUM_FRAMES)
    flat_sequence = flatten_pose_sequence(sequence)
    correlation = build_multiscale_correlation(flat_sequence)
    density, count, centers, target_source = build_density_target(metadata, sequence, category)

    output_dir = OUTPUT_BASE / category
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / json_path.with_suffix('.npz').name

    np.savez_compressed(
        output_path,
        sequence=flat_sequence.astype(np.float32),
        pose_sequence=sequence.astype(np.float32),
        temporal_correlation=correlation.astype(np.float32),
        density=density.astype(np.float32),
        count=np.array(count, dtype=np.float32),
        category_index=np.array(CATEGORY_TO_INDEX[category], dtype=np.int64),
        frame_count=np.array(raw_tensor.shape[0], dtype=np.int64),
        fps=np.array(float(metadata.get('fps', 0.0) or 0.0), dtype=np.float32),
    )

    return {
        'source_json': str(json_path.relative_to(PROJECT_ROOT)),
        'feature_file': str(output_path.relative_to(PROJECT_ROOT)),
        'category': category,
        'category_index': CATEGORY_TO_INDEX[category],
        'frame_count': int(raw_tensor.shape[0]),
        'count': float(count),
        'target_source': target_source,
        'pseudo_cycle_centers': centers,
    }


manifest = []
summary = Counter()

for category in CATEGORIES:
    input_dir = INPUT_BASE / category
    json_files = sorted(input_dir.glob('*.json'))
    if not json_files:
        print(f'No JSON files found for {category}: {input_dir}')
        continue

    for json_path in tqdm(json_files, desc=f'Hu pose features: {category}'):
        record = process_one_file(json_path, category)
        manifest.append(record)
        summary[category] += 1
        summary[record['target_source']] += 1

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print('Saved manifest:', MANIFEST_PATH)
print('Summary:', dict(summary))


## Validate Hu-Style Feature Files

The validation confirms that every feature file has the expected TransRAC-style pose shapes and contains finite numeric values.


In [ ]:
def validate_feature_file(path):
    errors = []
    with np.load(path) as data:
        expected_shapes = {
            'sequence': (NUM_FRAMES, EXPECTED_LANDMARKS * FEATURES_PER_LANDMARK),
            'pose_sequence': (NUM_FRAMES, EXPECTED_LANDMARKS, FEATURES_PER_LANDMARK),
            'temporal_correlation': (len(SCALES), NUM_FRAMES, NUM_FRAMES),
            'density': (NUM_FRAMES,),
        }
        for key, expected_shape in expected_shapes.items():
            if key not in data:
                errors.append(f'missing {key}')
                continue
            if data[key].shape != expected_shape:
                errors.append(f'{key} shape {data[key].shape} != {expected_shape}')
            if not np.isfinite(data[key]).all():
                errors.append(f'{key} contains non-finite values')
    return errors


feature_files = sorted(OUTPUT_BASE.glob('*/*.npz'))
validation_errors = {}
for feature_file in feature_files:
    errors = validate_feature_file(feature_file)
    if errors:
        validation_errors[str(feature_file.relative_to(PROJECT_ROOT))] = errors

print('feature files:', len(feature_files))
print('validation errors:', len(validation_errors))
if validation_errors:
    first_key = next(iter(validation_errors))
    print(first_key, validation_errors[first_key])
